# SFT datasets from baseline positives

Mine every **compliant** (strict CoT-control positive) rollout from the empty-prompt
baselines in `baselines/` into per-model SFT datasets under `sft/data/`.

Row format matches `cotcontrol.training.rendering` / `scripts/train_eval.py`:
`{"input": [...messages], "output": [assistant message], "meta": {...}}` — qwen
assistant content embeds `<think>...</think>`, gpt-oss uses thinking/text blocks.

**Split warning:** the original `baselines/<label>/` runs are on the canonical **test**
split. Datasets mined from them are suffixed `_testsplit` — fine for infra tests,
but training on them poisons any later test-split eval of that checkpoint. The
protocol-clean datasets come from the `<label>_train/` generations
(`baselines/run_baseline.py --split train`, same settings) and are suffixed `_train`.

In [1]:
import glob
import json
import os
from collections import Counter
from pathlib import Path

os.chdir(Path.cwd().parent if Path.cwd().name == "sft" else Path.cwd())  # repo root
assert Path("baselines").is_dir()

MODELS = {
    "gptoss20b": {"model": "openai/gpt-oss-20b", "family": "gptoss"},
    "gptoss120b": {"model": "openai/gpt-oss-120b", "family": "gptoss"},
    "qwen8b": {"model": "qwen/qwen3-8b", "family": "qwen"},
    "qwen32b": {"model": "qwen/qwen3-32b", "family": "qwen"},
}
OUT_DIR = Path("sft/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def sft_rows_from_result(result_path: str, label: str, family: str) -> list[dict]:
    """One SFT row per compliant, error-free, answer-bearing rollout."""
    r = json.loads(Path(result_path).read_text())
    split = r["config"].get("split")
    rows = []
    for rec in r["results"]:
        for s in rec["samples"]:
            if s["compliance"] != 1 or s["error"] or not s["output"] or not s["extracted_answer"]:
                continue
            reasoning, output = s["reasoning"] or "", s["output"]
            if family == "gptoss":
                assistant = {"role": "assistant", "content": [
                    {"type": "thinking", "thinking": reasoning},
                    {"type": "text", "text": output},
                ]}
            else:  # qwen
                assistant = {"role": "assistant",
                             "content": f"<think>\n{reasoning}\n</think>\n\n{output}"}
            rows.append({
                "input": rec["messages"],
                "output": [assistant],
                "meta": {
                    "label": label, "model": r["config"]["model"], "split": split,
                    "dataset": rec["dataset"], "id": rec["id"], "mode": rec["mode"],
                    "correct": s["correct"], "source": str(result_path),
                },
            })
    return rows


def build_datasets(dir_for_label, suffix: str) -> None:
    for label, info in MODELS.items():
        files = sorted(glob.glob(f"{dir_for_label(label)}/2026-*.json"))
        if not files:
            print(f"{label:11s} -> no generations at {dir_for_label(label)} (skipped)")
            continue
        rows = sft_rows_from_result(files[-1], label, info["family"])
        out = OUT_DIR / f"{label}_baseline_pos_{suffix}.jsonl"
        out.write_text("".join(json.dumps(r) + "\n" for r in rows))
        modes = Counter(r["meta"]["mode"] for r in rows)
        n_correct = sum(bool(r["meta"]["correct"]) for r in rows)
        print(f"{label:11s} -> {out}  n={len(rows)} (correct={n_correct})")
        print(f"{'':14s}per-mode: {dict(sorted(modes.items()))}")

## Test-split baselines (the original `baselines/<label>/` runs)

In [3]:
build_datasets(lambda label: f"baselines/{label}", "testsplit")

gptoss20b   -> sft/data/gptoss20b_baseline_pos_testsplit.jsonl  n=38 (correct=26)
              per-mode: {'multiple_word_suppression': 7, 'repeat_sentences': 7, 'word_suppression': 24}


gptoss120b  -> sft/data/gptoss120b_baseline_pos_testsplit.jsonl  n=184 (correct=95)
              per-mode: {'alternating_case': 2, 'ignore_question': 69, 'meow_between_words': 1, 'multiple_word_suppression': 17, 'repeat_sentences': 20, 'uppercase_thinking': 7, 'word_suppression': 68}


qwen8b      -> sft/data/qwen8b_baseline_pos_testsplit.jsonl  n=47 (correct=12)
              per-mode: {'lowercase_thinking': 7, 'meow_between_words': 25, 'multiple_word_suppression': 2, 'word_suppression': 13}


qwen32b     -> sft/data/qwen32b_baseline_pos_testsplit.jsonl  n=91 (correct=52)
              per-mode: {'ignore_question': 14, 'lowercase_thinking': 12, 'meow_between_words': 18, 'multiple_word_suppression': 7, 'word_suppression': 40}


## Train-split generations

Same empty-prompt rollouts on the canonical **train** split (safe to fine-tune on).
Generated 2026-08-26 with:
```bash
for m in "openai/gpt-oss-20b gptoss20b" "openai/gpt-oss-120b gptoss120b" \
         "qwen/qwen3-8b qwen8b" "qwen/qwen3-32b qwen32b"; do
  set -- $m
  .venv/bin/python baselines/run_baseline.py --model $1 --label $2 --split train --max-concurrency 150 &
done
```
Re-running this cell after generation finishes (files under `baselines/<label>_train/`)
builds the `_train` datasets.

In [4]:
build_datasets(lambda label: f"baselines/{label}_train", "train")

gptoss20b   -> sft/data/gptoss20b_baseline_pos_train.jsonl  n=35 (correct=19)
              per-mode: {'lowercase_thinking': 1, 'multiple_word_suppression': 5, 'repeat_sentences': 7, 'word_suppression': 22}


gptoss120b  -> sft/data/gptoss120b_baseline_pos_train.jsonl  n=161 (correct=73)
              per-mode: {'alternating_case': 1, 'ignore_question': 72, 'lowercase_thinking': 1, 'meow_between_words': 3, 'multiple_word_suppression': 9, 'repeat_sentences': 15, 'uppercase_thinking': 4, 'word_suppression': 56}


qwen8b      -> sft/data/qwen8b_baseline_pos_train.jsonl  n=38 (correct=11)
              per-mode: {'lowercase_thinking': 6, 'meow_between_words': 22, 'multiple_word_suppression': 2, 'word_suppression': 8}


qwen32b     -> sft/data/qwen32b_baseline_pos_train.jsonl  n=101 (correct=48)
              per-mode: {'ignore_question': 13, 'lowercase_thinking': 17, 'meow_between_words': 18, 'multiple_word_suppression': 10, 'word_suppression': 43}
